# Q5 — Language Modeling (WikiText-2)
**Models:** N-gram (CPU) → LSTM (GPU) → GPT-2 / DistilGPT2 (GPU)

**Dataset:** Full WikiText-2

**Runtime:** T4 GPU required

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q5'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Run N-gram Baseline (CPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=ngram \
        dataset.limit_train_samples=null \
        dataset.limit_validation_samples=null \
        dataset.limit_test_samples=null

In [ ]:
import glob, shutil
latest_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_ngram')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'N-gram results saved to Drive: {dest}')

## 3. Run LSTM (GPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=lstm \
        model.hidden_dim=256 \
        model.num_layers=2 \
        model.dropout=0.3 \
        model.max_epochs=15 \
        model.early_stopping_patience=3 \
        model.batch_size=32 \
        dataset.limit_train_samples=null \
        dataset.limit_validation_samples=null \
        dataset.limit_test_samples=null

In [ ]:
latest_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_lstm')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'LSTM results saved to Drive: {dest}')

## 4. Run GPT-2 / DistilGPT2 (GPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=gpt2 \
        model.model_name=distilgpt2 \
        model.eval_batch_size=8 \
        model.max_input_length=256 \
        dataset.limit_train_samples=null \
        dataset.limit_validation_samples=null \
        dataset.limit_test_samples=null

In [ ]:
latest_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_gpt2')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'GPT-2 results saved to Drive: {dest}')

## 5. Results Summary

In [ ]:
import json

print('=== Q5 Results Summary ===')
for run_dir in sorted(glob.glob('outputs/q5/run_*')):
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {os.path.basename(run_dir)} ---')
        print(json.dumps(metrics, indent=2, default=str)[:1500])